# 📊 Análise Exploratória de Dados (EDA) — Olist Recommender

> Notebook de apresentação do dataset processado para o sistema de recomendação.

**Projeto**: Olist Recommender System  
**Tech Challenge**: Fase 02 — Pós-graduação ML Engineering  
**Data**: 2026-06-18  
**Dataset**: `data/processed/interactions.parquet` (99.785 interações)

## 📋 Índice

1. [Configuração do Ambiente](#1)
2. [Carregamento dos Dados](#2)
3. [Estatísticas Básicas](#3)
4. [Distribuição do Target (review_score)](#4)
5. [Features Numéricas (price, freight_value)](#5)
6. [Features Categóricas](#6)
7. [Análise Temporal](#7)
8. [Comportamento de Usuários](#8)
9. [Comportamento de Produtos](#9)
10. [Padrões de Interação](#10)
11. [Análise Bivariada](#11)
12. [Sparsity da Matriz User-Item](#12)
13. [Conclusões](#13)

<a id="1"></a>
## 1. ⚙️ Configuração do Ambiente

Importação das bibliotecas e configuração de paths.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from loguru import logger
import matplotlib
matplotlib.use('Agg')  # Para salvar figuras, mas inline funciona em Jupyter
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Configuração
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['figure.max_open_warning'] = 50  # Suprimir warnings de muitas figuras

# Paths
DATA_PATH = Path('../data/processed/interactions.parquet')
FIGURES_DIR = Path('../reports/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


<a id="2"></a>
## 2. 📥 Carregamento dos Dados

Carregamos o dataset processado de interações user-item (99.785 linhas, 10 colunas) gerado pelo pipeline `src/data_preparation.py`.

In [ ]:
logger.info(f"Carregando dados de: {DATA_PATH}")
df = pd.read_parquet(DATA_PATH)
logger.info(f"Shape do dataset: {df.shape}")
display(df.head())
df.info()


<a id="3"></a>
## 3. 📐 Estatísticas Básicas

### 3.1 Visão Geral

In [ ]:
mem_usage = df.memory_usage(deep=True).sum() / (1024**2)
n_duplicates = int(df.duplicated().sum())

print(f"Shape: {df.shape}")
print(f"Linhas duplicadas: {n_duplicates}")
print(f"Uso de memória: {mem_usage:.2f} MB")


### 3.2 Tipos de Dados

In [ ]:
display(df.dtypes.astype(str).value_counts().to_frame('count'))


### 3.3 Valores Nulos

In [ ]:
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100)
missing_df = pd.DataFrame({'count': missing_count, 'pct': missing_pct})
display(missing_df[missing_df['count'] > 0])

plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')
plt.title('Valores Nulos (Heatmap)')
plt.tight_layout()
plt.show()


<a id="4"></a>
## 4. 🎯 Distribuição do Target (review_score)

A variável `review_score` representa a satisfação do cliente (1 a 5). Vamos entender sua distribuição.

In [ ]:
display(df['review_score'].describe().to_frame('review_score_stats'))
display(df['review_score'].value_counts(dropna=False).sort_index().to_frame('count'))

positivas = df['review_score'] >= 4
negativas = df['review_score'] <= 2
neutras = df['review_score'] == 3
ausentes = df['review_score'].isna()

total = len(df)
pct_pos = positivas.sum() / total * 100
pct_neg = negativas.sum() / total * 100
pct_neu = neutras.sum() / total * 100
pct_aus = ausentes.sum() / total * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.countplot(data=df.dropna(subset=['review_score']), x='review_score', ax=axes[0], palette='viridis')
axes[0].set_title('Distribuição de Review Score')
axes[0].set_xlabel('Review Score')
axes[0].set_ylabel('Contagem')

labels = ['Positivas', 'Neutras', 'Negativas', 'Ausentes']
sizes = [pct_pos, pct_neu, pct_neg, pct_aus]
colors = sns.color_palette('viridis', 4)
axes[1].pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Sentimento das Avaliações')

plt.tight_layout()
fig.savefig(FIGURES_DIR / '01_review_score_distribution.png')
plt.show()


<a id="5"></a>
## 5. 💰 Features Numéricas (price, freight_value)

Ambas apresentam distribuição log-normal. Vamos visualizar.

In [ ]:
features = ['price', 'freight_value']
for feat in features:
    if feat in df.columns:
        percentiles = [0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
        stats = df[feat].describe(percentiles=percentiles)
        print(f"--- Estatísticas para {feat} ---")
        print(stats)
        
        Q1 = df[feat].quantile(0.25)
        Q3 = df[feat].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = df[(df[feat] < lower_bound) | (df[feat] > upper_bound)]
        pct_outliers = len(outliers) / len(df) * 100
        print(f"% outliers para {feat}: {pct_outliers:.2f}%\n")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

if 'price' in df.columns:
    p99_price = df['price'].quantile(0.99)
    sns.histplot(df[df['price'] <= p99_price]['price'], bins=50, ax=axes[0, 0], color='skyblue')
    axes[0, 0].set_title('Histograma de Price (Cap em p99)')
    
    sns.boxplot(x=df['price'], ax=axes[0, 1], color='skyblue')
    axes[0, 1].set_title('Boxplot de Price')
    
if 'freight_value' in df.columns:
    p99_freight = df['freight_value'].quantile(0.99)
    sns.histplot(df[df['freight_value'] <= p99_freight]['freight_value'], bins=50, ax=axes[1, 0], color='lightgreen')
    axes[1, 0].set_title('Histograma de Freight Value (Cap em p99)')
    
    sns.boxplot(x=df['freight_value'], ax=axes[1, 1], color='lightgreen')
    axes[1, 1].set_title('Boxplot de Freight Value')

plt.tight_layout()
fig.savefig(FIGURES_DIR / '02_numerical_distributions.png')
plt.show()


<a id="6"></a>
## 6. 🏷️ Features Categóricas

### 6.1 Top Categorias

In [ ]:
cat_col = 'product_category_name_english'
if cat_col in df.columns:
    counts = df[cat_col].value_counts()
    total = len(df[cat_col].dropna())
    
    top15 = counts.head(15)
    top15_pct = top15 / total * 100
    display(top15_pct.to_frame('% do Total (Top 15)'))
    
    top5_pct = counts.head(5).sum() / total * 100
    top10_pct = counts.head(10).sum() / total * 100
    print(f"% acumulada Top 5: {top5_pct:.2f}%")
    print(f"% acumulada Top 10: {top10_pct:.2f}%")
    
    rare_cats = (counts < 10).sum()
    print(f"Número de categorias raras (<10 interações): {rare_cats}")


### 6.2 Tipo de Pagamento

In [ ]:
pay_col = 'payment_type'
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

if cat_col in df.columns:
    sns.barplot(x=top15.values, y=top15.index, ax=axes[0], palette='viridis')
    axes[0].set_title('Top 15 Categorias de Produtos')
    axes[0].set_xlabel('Contagem')

if pay_col in df.columns:
    pay_counts = df[pay_col].value_counts()
    pay_pct = pay_counts / len(df[pay_col].dropna()) * 100
    display(pay_pct.to_frame('% por Tipo de Pagamento'))
    
    sns.barplot(x=pay_counts.index, y=pay_counts.values, ax=axes[1], palette='Set2')
    axes[1].set_title('Distribuição de Tipos de Pagamento')
    axes[1].set_ylabel('Contagem')
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
fig.savefig(FIGURES_DIR / '03_categorical_distribution.png')
plt.show()


<a id="7"></a>
## 7. 📅 Análise Temporal

O dataset cobre o período de 2016-09 a 2018-08.

In [ ]:
time_col = 'order_purchase_timestamp'
if time_col in df.columns:
    if not pd.api.types.is_datetime64_any_dtype(df[time_col]):
        df[time_col] = pd.to_datetime(df[time_col])
        
    min_date = df[time_col].min()
    max_date = df[time_col].max()
    range_days = (max_date - min_date).days
    print(f"Data mínima: {min_date}, máxima: {max_date}, range: {range_days} dias")
    
    df['year'] = df[time_col].dt.year
    df['month'] = df[time_col].dt.month
    df['year_month'] = df[time_col].dt.to_period('M').astype(str)
    df['day_of_week'] = df[time_col].dt.day_name()
    df['hour'] = df[time_col].dt.hour
    
    year_dist = df['year'].value_counts(normalize=True) * 100
    print(f"\nDistribuição por ano:\n{year_dist}")
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    
    # 1. Série temporal por mês
    monthly_counts = df['year_month'].value_counts().sort_index()
    sns.lineplot(x=monthly_counts.index, y=monthly_counts.values, ax=axes[0, 0], marker='o')
    axes[0, 0].set_title('Interações por Mês')
    axes[0, 0].tick_params(axis='x', rotation=45)
    
    # 2. Barplot por dia da semana
    order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    sns.countplot(data=df, x='day_of_week', order=order, ax=axes[0, 1], palette='viridis')
    axes[0, 1].set_title('Interações por Dia da Semana')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # 3. Barplot por hora do dia
    sns.countplot(data=df, x='hour', ax=axes[1, 0], color='skyblue')
    axes[1, 0].set_title('Interações por Hora do Dia')
    
    # 4. Heatmap ano x mês
    heatmap_data = df.groupby(['year', 'month']).size().unstack(fill_value=0)
    sns.heatmap(heatmap_data, cmap='YlGnBu', annot=True, fmt='d', ax=axes[1, 1])
    axes[1, 1].set_title('Heatmap Ano x Mês')
    
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / '04_temporal_distribution.png')
    plt.show()


<a id="8"></a>
## 8. 👤 Comportamento de Usuários

Análise da distribuição de interações por usuário (cold-start).

In [ ]:
user_col = 'customer_unique_id'
if user_col in df.columns:
    interactions_user = df[user_col].value_counts()
    
    percentiles = [0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
    stats_user = interactions_user.describe(percentiles=percentiles)
    display(stats_user.to_frame('Interações por Usuário'))
    
    bins = [0, 1, 2, 3, 4, np.inf]
    labels = ['1', '2', '3', '4', '5+']
    cold_start_user = pd.cut(interactions_user, bins=bins, labels=labels).value_counts(normalize=True) * 100
    cold_start_user = cold_start_user.sort_index()
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    sns.histplot(interactions_user, bins=50, log_scale=(False, True), ax=axes[0], color='salmon')
    axes[0].set_title('Distribuição de Interações por Usuário (Log Scale)')
    axes[0].set_xlabel('Número de Interações')
    
    sns.barplot(x=cold_start_user.index, y=cold_start_user.values, ax=axes[1], palette='viridis')
    axes[1].set_title('Distribuição de Cold Start - Usuários')
    axes[1].set_ylabel('% de Usuários')
    
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / '05_user_behavior.png')
    plt.show()


<a id="9"></a>
## 9. 📦 Comportamento de Produtos

Análise da distribuição de interações por produto (long-tail).

In [ ]:
item_col = 'product_id'
if item_col in df.columns:
    interactions_item = df[item_col].value_counts()
    
    percentiles = [0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
    stats_item = interactions_item.describe(percentiles=percentiles)
    display(stats_item.to_frame('Interações por Produto'))
    
    bins = [0, 1, 5, 10, 50, np.inf]
    labels = ['1', '2-5', '6-10', '11-50', '51+']
    cold_start_item = pd.cut(interactions_item, bins=bins, labels=labels).value_counts(normalize=True) * 100
    cold_start_item = cold_start_item.sort_index()
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    sns.histplot(interactions_item, bins=50, log_scale=(False, True), ax=axes[0], color='orchid')
    axes[0].set_title('Distribuição de Interações por Produto (Log Scale)')
    axes[0].set_xlabel('Número de Interações')
    
    sns.barplot(x=cold_start_item.index, y=cold_start_item.values, ax=axes[1], palette='Set2')
    axes[1].set_title('Distribuição de Cold Start - Produtos')
    axes[1].set_ylabel('% de Produtos')
    
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / '06_product_behavior.png')
    plt.show()


<a id="10"></a>
## 10. 🔄 Padrões de Interação

Análise de `purchase_count` e `has_review`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if 'purchase_count' in df.columns:
    display(df['purchase_count'].describe().to_frame('purchase_count'))
    
    repetidas = (df['purchase_count'] > 1).sum() / len(df) * 100
    print(f"% de interações que são compras repetidas: {repetidas:.2f}%")
    
    df_plot = df.copy()
    df_plot['purchase_count_cap'] = df_plot['purchase_count'].clip(upper=5)
    sns.countplot(data=df_plot, x='purchase_count_cap', ax=axes[0], palette='viridis')
    axes[0].set_title('Distribuição de Purchase Count (Cap em 5)')
    axes[0].set_xticklabels(['1', '2', '3', '4', '5+'])

if 'has_review' in df.columns:
    review_dist = df['has_review'].value_counts(normalize=True) * 100
    
    axes[1].pie(review_dist.values, labels=review_dist.index, autopct='%1.1f%%', colors=sns.color_palette('Set2', 2), startangle=90)
    axes[1].set_title('Distribuição de Has Review')
    
plt.tight_layout()
fig.savefig(FIGURES_DIR / '07_interaction_patterns.png')
plt.show()


<a id="11"></a>
## 11. 🔗 Análise Bivariada

Relações entre variáveis principais.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if 'price' in df.columns and 'review_score' in df.columns:
    df_plot = df.dropna(subset=['price', 'review_score']).copy()
    df_plot['price_log'] = np.log1p(df_plot['price'])
    
    sns.boxplot(data=df_plot, x='review_score', y='price_log', ax=axes[0], palette='viridis')
    axes[0].set_title('Boxplot: Price (Log) vs Review Score')
    
    corr = df_plot[['price', 'review_score']].corr().iloc[0, 1]
    print(f"Correlação Pearson price vs review_score: {corr:.4f}")

if 'purchase_count' in df.columns and 'has_review' in df.columns:
    crosstab = pd.crosstab(df['purchase_count'].clip(upper=5), df['has_review'], normalize='index') * 100
    
    crosstab.plot(kind='bar', stacked=True, ax=axes[1], color=sns.color_palette('Set2', 2))
    axes[1].set_title('Purchase Count vs Has Review')
    axes[1].set_xlabel('Purchase Count (Capped at 5)')
    axes[1].set_ylabel('%')
    axes[1].set_xticklabels(['1', '2', '3', '4', '5+'], rotation=0)

plt.tight_layout()
fig.savefig(FIGURES_DIR / '08_bivariate_analysis.png')
plt.show()


<a id="12"></a>
## 12. 📊 Sparsity da Matriz User-Item

A esparsidade da matriz afeta diretamente a performance dos modelos de recomendação.

In [ ]:
user_col = 'customer_unique_id'
item_col = 'product_id'

if user_col in df.columns and item_col in df.columns:
    n_users = df[user_col].nunique()
    n_products = df[item_col].nunique()
    n_interactions = len(df)
    
    possible_interactions = n_users * n_products
    actual_density = n_interactions / possible_interactions
    sparsity = 1 - actual_density
    
    print(f"Usuários únicos: {n_users}")
    print(f"Produtos únicos: {n_products}")
    print(f"Interações totais: {n_interactions}")
    print(f"Interações possíveis: {possible_interactions}")
    print(f"Densidade real: {actual_density:.8f}")
    print(f"Sparsity: {sparsity:.8f}\n")
    
    print("--- Benchmark ---")
    print("MovieLens 1M: Sparsity ~95.5%")
    print("Amazon Reviews: Sparsity >99.9%")


<a id="13"></a>
## 13. ✅ Conclusões e Próximos Passos

### Principais Descobertas
- Dataset com 99.785 interações, 93.358 usuários e 32.216 produtos
- Sparsity de 99.99% (similar a Amazon Reviews)
- Cold-start severo em usuários (90%+ fizeram apenas 1 compra)
- Long-tail em produtos (50%+ com apenas 1 compra)
- review_score majoritariamente positivo (viés de quem compra = gostou)

### Implicações para Modelagem
- Necessidade de modelos com embeddings (para cold-start)
- Fallback de popularidade para novos usuários/produtos
- Split temporal é mandatório (evitar data leakage)

### Próximas Etapas
1. Split temporal (train/val/test)
2. Baselines (Sklearn): Popularidade, Item-Item CF, SVD
3. MLP/NCF (PyTorch)
4. Pipeline DVC
5. Tracking MLflow